写这系列文章的初衷是夏洛铁烦恼和《万物发明指南》带来的灵感，希望能帮助到大家。不管你穿越回过去的哪个时间点，你总要有点真本事，才能在这个快节奏的世界里保持领先。我在想如果我穿回过去，我只知道AI会来，但是不知道AI是怎么来的，我想这就是我写这系列文章的初衷。当前AI巨变的基座就是Transformer，你提前把论文发出来，就能让Transformer的作者感觉活在你的阴影之下。（哈哈哈，烂梗）

基础python语法

In [ ]:
import torch
import torch.nn as nn
import torch.utils.data as data
import math
import numpy as np

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"设备: {device}")

字典推导式

In [29]:
idx2src = {v: k
           for k, v in src_vocab.items()}
print(idx2src)


{0: '<pad>', 1: '<sos>', 2: '<eos>', 3: 'world'}


In [30]:
text = "I love ai"
words = text.split()
print(words)

['I', 'love', 'ai']


In [31]:
chars = ["我", "爱", "AI"]
result = "".join(chars)
print(result)





我爱AI


Pytorch 张量操作


tensor() 是从数据创建

In [2]:
import torch

# 从列表创建
x = torch.tensor([1, 2, 3])
print(x)
print(x.shape)

# 从二维列表创建
matrix = torch.tensor([[1, 2], [3, 4]])
print(matrix)
print(matrix.shape)


tensor([1, 2, 3])
torch.Size([3])
tensor([[1, 2],
        [3, 4]])
torch.Size([2, 2])


zeros() 创建全0张量
ones() 创建全1张量
randn() 创建标准正态分布张量

In [33]:
x = torch.zeros(2, 3)
print(x)
print(x.shape)

x = torch.ones(3, 2)
print(x)
print(x.shape)

x = torch.randn(3, 4)
print(x)
print(x.shape)


tensor([[0., 0., 0.],
        [0., 0., 0.]])
torch.Size([2, 3])
tensor([[1., 1.],
        [1., 1.],
        [1., 1.]])
torch.Size([3, 2])
tensor([[ 1.0058, -0.1344, -0.6025,  2.9607],
        [-0.0799, -0.1975, -0.0773, -1.0474],
        [ 0.3114, -2.2809, -0.6320,  0.8232]])
torch.Size([3, 4])


In [34]:
x = torch.randn(2, 4, 5)
print(x.shape)
print(x.size())

# 维度
print(x.dim())
# 元素总数
print(x.numel())

#数据类型
print(x.dtype)



torch.Size([2, 4, 5])
torch.Size([2, 4, 5])
3
40
torch.float32


张量索引

In [11]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
# 访问单个元素
print(f"element at row 0, column 1: {x[0, 1]}")

#访问第一行
print(f"first row: {x[0]}")

#切片
print(f"all rows, column 1 to end: {x[:, 1:]}")
print(f"all rows, column 0 to end-1: {x[:, :-1]}")

element at row 0, column 1: 2
first row: tensor([1, 2, 3])
all rows, column 1 to end: tensor([[2, 3],
        [5, 6]])
all rows, column 0 to end-1: tensor([[1, 2],
        [4, 5]])


张量形状操作

In [35]:
x = torch.randn(2, 3, 4)
print(x.shape)

# 变换维度
x_flat = x.view(2, -1)
print(x_flat)
print(x_flat.shape)

# 可自动推断，-1可在任何位置，但只能出现一次
x_high = x_flat.view(2, 2, -1, 2)
print(x_high)
print(x_high.shape)

x_flat = x.view(-1)
print(x_flat)
print(x_flat.shape)


torch.Size([2, 3, 4])
tensor([[ 1.1022,  0.6759,  0.3059,  0.6087, -0.5389, -1.4465,  0.7620,  0.8301,
         -0.2399, -0.6780,  0.5297, -0.2110],
        [ 0.2521,  0.1959, -0.7602, -3.2475,  0.2208, -1.3855,  0.7900,  0.4169,
         -1.4289,  1.7383, -1.2302,  0.9859]])
torch.Size([2, 12])
tensor([[[[ 1.1022,  0.6759],
          [ 0.3059,  0.6087],
          [-0.5389, -1.4465]],

         [[ 0.7620,  0.8301],
          [-0.2399, -0.6780],
          [ 0.5297, -0.2110]]],


        [[[ 0.2521,  0.1959],
          [-0.7602, -3.2475],
          [ 0.2208, -1.3855]],

         [[ 0.7900,  0.4169],
          [-1.4289,  1.7383],
          [-1.2302,  0.9859]]]])
torch.Size([2, 2, 3, 2])
tensor([ 1.1022,  0.6759,  0.3059,  0.6087, -0.5389, -1.4465,  0.7620,  0.8301,
        -0.2399, -0.6780,  0.5297, -0.2110,  0.2521,  0.1959, -0.7602, -3.2475,
         0.2208, -1.3855,  0.7900,  0.4169, -1.4289,  1.7383, -1.2302,  0.9859])
torch.Size([24])


什么时候使用呢？拆分多头注意力的时候

In [36]:
B, T, C = 2, 3, 8
x = torch.randn(B, T, C)

n_heads = 2
head_dim = C // n_heads

x_heads = x.view(B, T, n_heads, head_dim)
print(x_heads)
print(x_heads.shape)


tensor([[[[ 1.6161,  0.1314,  0.9200,  0.2275],
          [-0.8699, -0.3339, -0.0678,  1.0133]],

         [[-0.4368,  2.3668, -0.3423, -0.4266],
          [ 0.5844, -0.1753, -0.8172, -1.2363]],

         [[ 0.1089, -0.4866,  0.7229,  1.9353],
          [-0.6481,  0.5596, -1.3915, -0.1506]]],


        [[[ 1.2013,  0.7214, -0.1697,  0.3922],
          [ 0.1120, -0.7146, -0.4540, -0.3829]],

         [[-0.9626,  0.1295,  0.0538, -0.7232],
          [ 1.5243,  0.8490,  0.0570, -0.3736]],

         [[ 0.9435, -0.0927, -0.3869, -0.2661],
          [-0.2799,  1.3702,  0.2367,  0.2634]]]])
torch.Size([2, 3, 2, 4])


transpose() 交换维度

In [15]:

x = torch.randn(2, 3, 4)
print(f"original shape: {x.shape}")

# 交换第1维和第2维，注意：维度是从0开始计数的
x_heads = x.transpose(1, 2)
print(f"after transpose: {x_heads.shape}")


original shape: torch.Size([2, 3, 4])
after transpose: torch.Size([2, 4, 3])


permute() 重新排列所有维度

In [17]:
x = torch.randn(2, 3, 4, 5)
print(f"original shape: {x.shape}")

x_perm = x.permute(0, 3, 1, 2)
print(f"after permute: {x_perm.shape}")

original shape: torch.Size([2, 3, 4, 5])
after permute: torch.Size([2, 5, 3, 4])


你会不会有疑惑，为什么会有permute和transpose 这两个看起来类似的函数呢？
因为transpose是做矩阵转置的，只交换两个维度，不需要感知其他维度，而permute可以重排列所有维度，但是如果需要仅做转置的时候，transpose更方便；不需要写成 permute(1,0,2,3,4,...) 这样的形式，如果维度很多，就会很长。

下面是真实transformer可能使用的方法：


In [42]:
B, T, C = 2, 3, 512
n_head = 8
head_dim = C // n_head

x = torch.randn(B, T, C)

x = x.view(B, T, n_head, head_dim)

x = x.permute(0, 2, 1, 3)
print(x.shape)


torch.Size([2, 8, 3, 64])


unsqueeze() 增加维度，解压缩

In [19]:
import torch

x = torch.tensor([1, 2, 3])
print(f"original shape: {x.shape}")

x1 = x.unsqueeze(0)
print(f"after unsqueeze at dim 0: {x1.shape}")

x2 = x.unsqueeze(1)
print(f"after unsqueeze at dim 1: {x2.shape}")


original shape: torch.Size([3])
after unsqueeze: torch.Size([1, 3])
after unsqueeze: torch.Size([3, 1])


squeeze() 删除/压缩  仅大小为1的维度

In [20]:
x = torch.randn(1, 3, 1, 4)
print(f"original shape: {x.shape}")

x = x.squeeze()
print(f"after squeeze: {x.shape}")

original shape: torch.Size([1, 3, 1, 4])
after squeeze: torch.Size([3, 4])


为什么有了view还要有squeeze和unsqueeze呢？

如果你想用view在某个位置上加一个维度，你要写成
x.view(x.size(0), 1, x.size(1), x.size(2),...)
而unsqeeuze只要写成下面的形式，不需要关心其他维度的size，而view必须声明所有其他不想变维度的值。
x.unsqueeze(1)



torch.cat() 拼接张量


In [21]:
x1 = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(x1.shape)

x2 = torch.tensor([[7, 8, 9], [10, 11, 12]])
print(x2.shape)

# 在第0维拼接
x_cat = torch.cat([x1, x2], dim=0)
print(f"after cat at dim 0: {x_cat.shape}")

# 在第1维拼接
x_cat = torch.cat([x2, x1], dim=1)
print(f"after cat at dim 1: {x_cat.shape}")


torch.Size([2, 3])
torch.Size([2, 3])
after cat at dim 0: torch.Size([4, 3])
after cat at dim 1: torch.Size([2, 6])


contiguous() 确保张量在内存中是连续的，某些操作，比如转置 transpose, permute等操作后改变的是内存张量的布局，然而view又要求内存必须是连续的，所以调用contiguous确保内存连续。

In [50]:
x = torch.randn(2, 3, 4)
print(x.shape)

x = x.transpose(0, 1)
print(x.shape)

# 报错
# x = x.view(2, 4, 3)
# print(x.shape)

# 内存连续后无报错
x = x.contiguous()
x = x.view(2, 4, 3)
print(x.shape)


torch.Size([2, 3, 4])
torch.Size([3, 2, 4])
torch.Size([2, 4, 3])


# 神经网络模块
nn.Module 所有神经网络的基类，有点类似于Java的Object类，所有神经网络模块都继承自它。

python 继承的写法
```
class SubClass(SuperClass):
    def __init__(self, ...):
        super().__init__(...)
        ...
```

In [23]:
import torch
import torch.nn as nn


class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()

        self.linear = nn.Linear(10, 5)

    def forward(self, x):
        return self.linear(x)


model = MyModel()

print(type(model))

print(isinstance(model, MyModel))

<class '__main__.MyModel'>
True


In [24]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)

super().__init() 是必须调用的，这是为了初始化父类 nn.Module
具体可以带你进去 module文件中的__init__() 方法去看下，里面做了很多初始化工作，比如parameters和buffer的注册，。
注册了才能能支持 train（）和eval()方法。
- 支持to方法，用于将模型移动到CPU或GPU

In [26]:
class BrokenModel(nn.Module):
    def __init__(self):
        self.linear = nn.Linear(10, 5)

    def forward(self, x):
        return self.linear(x)


model = BrokenModel()

AttributeError: cannot assign module before Module.__init__() call

forward()函数 看名字就知道是定义了数据如何在网络中流动的，从input到output的过程。既然有前向传播，那什么是后向传播呢？就是通过损失计算梯度，根据梯度更新参数的过程。


In [5]:
import torch
import torch.nn as nn


class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 5)

    def forward(self, x):
        """
        定义前向传播过程，从 x -> fc1 -> relu -> fc2 -> output
        """
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x


model = MyModel()
x = torch.randn(2, 10)
output = model(x)  # 这里会自动调用call方法，call方法里面调用了forward方法
print(output.shape)


torch.Size([2, 5])


那么问题来了，为什么不直接调用forward方法呢？


In [ ]:
# nn.Module的简化实现版本（实际更复杂），此代码仅做展示，不需要运行。
class Module:
    def __call__(self, *args, **kwargs):
        # 1. 执行一些前置操作
        # 2. 调用forward方法
        result = self.forward(*args, **kwargs)
        # 3. 执行一些后置操作
        return result

    def forward(self, *args, **kwargs):
        raise NotImplementedError("子类必须实现 forward 方法")


# 举例来说，nn.Linear里面就有forward方法，下面是一个简化实现版本，仅供展示，不需要运行。
class Linear(Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))

    def forward(self, x):
        return x @ self.weight.t() + self.bias


#真正使用的时候
linear = nn.Linear(10, 5)
x = torch.randn(2, 10)

output = linear(x)  # 这里会自动调用call方法，call方法里面调用了forward方法

下面用一个简单的实例说明整个调用链是怎样的，包括自己实现一个简单的层和包含多层的模型。

In [16]:
import torch
import torch.nn as nn


# 先定一个简单的层，这个层只有weight，没有bias
class MyLayer(nn.Module):
    def __init__(self, in_features, out_features, name="MyLayer"):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.name = name
        # 并且你还可以在这里添加你自己想要的逻辑，比如我这里打印一个初始化完成
        print(f"MyLayer 初始化完成，weight shape: {self.weight.shape}")

    def forward(self, x):
        # 这里可以添加一些其他的操作，比如我这里打印输入的形状
        print(f"MyLayer {self.name} 的forward被调用， input shape: {x.shape}")
        return x @ self.weight.t()


# 再定义一个模型，包含这个层
class MyModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.layer1 = MyLayer(in_features, 40, "layer1")
        self.layer2 = MyLayer(40, 5, "layer2")
        print(f"MyModel 初始化完成，layer1: {self.layer1.weight.shape}，layer2: {self.layer2.weight.shape}")

    def forward(self, x):
        print(f"MyModel 的forward被调用， input shape: {x.shape}")
        x = self.layer1(x)
        x = torch.relu(x)
        return self.layer2(x)


print("====创建模型====")
model = MyModel(10, 20)

print("\n====调用模型====")
x = torch.randn(2, 10)
output = model(x)

print(f"\n最终输出 shape: {output.shape}")


====创建模型====
MyLayer 初始化完成，weight shape: torch.Size([40, 10])
MyLayer 初始化完成，weight shape: torch.Size([5, 40])
MyModel 初始化完成，layer1: torch.Size([40, 10])，layer2: torch.Size([5, 40])

====调用模型====
MyModel 的forward被调用， input shape: torch.Size([2, 10])
MyLayer layer1 的forward被调用， input shape: torch.Size([2, 10])
MyLayer layer2 的forward被调用， input shape: torch.Size([2, 40])

最终输出 shape: torch.Size([2, 5])


所以你应该能知道调用链路是啥了吧！自己绘制一下呢！

```
model(x) -> model.__call__(x) -> model.forward(1) -> self.layer1(x)
 -> layer1.__call__(x) -> layer1.forward(x) -> relu(x) -> self.layer2(x)
 -> layer2.__call__(x) -> layer2.forward(x)
```

# 一些常用的层

In [18]:
# nn.Linear 就是最简单的全连接层，线性乘法加上偏置项 y = wx + b
linear = nn.Linear(10, 5)

x = torch.randn(2, 10)
output = linear(x)
print(f"linear output shape: {output.shape}")

# nn.Embedding 词嵌入层，将离散的词索引映射到连续的向量空间
# 词大小为 (1000, 50)，表示有 1000 个词，每个词用 50 维向量表示
embedding = nn.Embedding(num_embeddings=1000, embedding_dim=64)

# Embedding相当于只做了查表操作，没有任何计算，就是取出了对应索引的向量，如上就是一个 64 维的向量
# 单个输入词的索引
indices = torch.tensor([1, 5, 3, 10])
output = embedding(indices)
print(f"embedding output shape: {output.shape}")  # (4, 64)

# batch 输入
batch_indices = torch.tensor([[1, 4, 6], [10, 5, 19]])
output = embedding(batch_indices)
print(f"embedding batch output shape: {output.shape}")  # (2, 3, 64)




linear output shape: torch.Size([2, 5])
embedding output shape: torch.Size([4, 64])
embedding batch output shape: torch.Size([2, 3, 64])


In [32]:
# 1. 设置随机种子，保证每次运行结果一致，方便观察
torch.manual_seed(42)
# 定义输入：[Batch=2, Sequence=2, Feature=5]
# 这里把维度设小一点（5），方便肉眼看数据
dim = 5
x = torch.randn(2, 2, dim)

# nn.LayerNorm 层归一化,对每个样本的特征进行归一化，并不是说把特征值都归一化到 [-1, 1] 之间，
# 而是将其归一化到均值为0，方差为1的分布。
layer_norm = nn.LayerNorm(dim)

x = torch.randn(2, 10, dim)
print(f"layer_norm x[0][0]: {x[0][0]}")  # tensor([ 0.3189, -0.4245,  0.3057, -0.7746,  0.0349])
print(
    f"layer_norm x[0][0] normalized: {layer_norm(x[0][0])}")  # tensor([ 0.9959, -0.7387,  0.9651, -1.5555,  0.3333], grad_fn=<NativeLayerNormBackward0>)

# nn.Dropout 随机将一些参数设置为0，避免过拟合，只在训练时启用，在推理时禁用，为什么过拟合要避免呢？
# 举一个炒股的例子，比如你的训练数据是股灾时候的数据，而测试数据正好是大牛市，

# 这里是说有50%的概率会被设为0，这里只是为了演示，设置的大了一点
drop_out = nn.Dropout(0.5)

print(f"drop_out x[0][0]: {x[0][0]}")  # tensor([ 0.3189, -0.4245,  0.3057, -0.7746,  0.0349])
print(f"drop_out x[0][0] dropped: {drop_out(x[0][0])}")  # tensor([ 0.0000, -0.0000,  0.6114, -1.5492,  0.0000])




layer_norm x[0][0]: tensor([ 0.3189, -0.4245,  0.3057, -0.7746,  0.0349])
layer_norm x[0][0] normalized: tensor([ 0.9959, -0.7387,  0.9651, -1.5555,  0.3333],
       grad_fn=<NativeLayerNormBackward0>)
drop_out x[0][0]: tensor([ 0.3189, -0.4245,  0.3057, -0.7746,  0.0349])
drop_out x[0][0] dropped: tensor([ 0.0000, -0.0000,  0.6114, -1.5492,  0.0000])


这里聪明的你已经发现了，为什么dropout的非0值变化了呢，你刚才不是说只会有一些值被置为0吗？

那你观察总结下其他值的规律，有没有什么发现，是不是他们变成了原来的2倍？
被丢掉的值凭空消失了对吧，那把对应的非0值放大一倍，就能保证这些特征期望不变。缩放的倍数是 1/(1-p)，其中 p 是 dropout 的概率。

In [35]:
# nn.ModuleList 其实就是一个列表，里面保存了多个子层。

layers = nn.ModuleList([
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
])

# 激活函数
# 常用的激活函数有 ReLU, Sigmoid, Tanh, Softmax 等，也需要根据场景选择。
import torch.nn.functional as F

# ReLU，relu是在正负半轴都是线性的，负数会被置为0，正数保持不变。

x = torch.tensor([-1.0, -0.5, 1.0, 2.0])
output = F.relu(x)
print(f"relu output: {output}")  # tensor([0., 0., 1., 2.])

# softmax一般用于最后的分类任务，将输出转换为概率分布。
x = torch.tensor([1.0, 2.0, 3.0])
output = F.softmax(x, dim=-1)
print(f"softmax output: {output}")  # tensor([0.0900, 0.2447, 0.6652])
print(output.sum())



relu output: tensor([0., 0., 1., 2.])
softmax output: tensor([0.0900, 0.2447, 0.6652])
tensor(1.)


# 训练相关的内容

设备管理 - to(device)

In [36]:
# 如我们前面提到的设备管理

#检测设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"using device: {device}")

# 转移张量到设备，这是什么意思呢？
# 之前数据都在内存中
x = x.to(device)

# 转移模型到设备上
model.to(device)

# 聪明的小朋友又发现了，为什么上面的x 需要赋值呢，而model不需要呢？
# 可以参考 https://pytorch.zhangxiann.com/7-mo-xing-qi-ta-cao-zuo/7.3-shi-yong-gpu-xun-lian-mo-xing 这篇文章的内容
# inplace定义：原地操作/
# tensor是因为从内存转移到了显存，所以地址不一样了，不是inplace操作，需要有一个新的变量来接收，当然用原来的变量也是可以的，比如我上面就是用x来接收的
# 而model的to方法，是将参数tensor都做了设备转移（上面说了参数肯定不是inplace），但是model会将gpu的tensor替换原本的model中指向cpu的tensor，虽然内部对应的参数换了，但model这个容器本身还是同一个对象引用。

x_cpu = torch.ones((3, 3))
print("x_cpu:\ndevice: {} is_cuda: {} id: {}".format(x_cpu.device, x_cpu.is_cuda, id(x_cpu)))

x_gpu = x_cpu.to(device)
print("x_gpu:\ndevice: {} is_cuda: {} id: {}".format(x_gpu.device, x_gpu.is_cuda, id(x_gpu)))

net = nn.Sequential(nn.Linear(3, 3))

print("\nid:{} is_cuda: {}".format(id(net), next(net.parameters()).is_cuda))

net.to(device)
print("\nid:{} is_cuda: {}".format(id(net), next(net.parameters()).is_cuda))



using device: mps
x_cpu:
device: cpu is_cuda: False id: 13540424976
x_gpu:
device: mps:0 is_cuda: False id: 13547558608

id:13539200464 is_cuda: False

id:13539200464 is_cuda: False


损失函数

In [ ]:
# 损失函数有很多种，看你的任务是分类还是回归，你需要选择不同的损失函数。
# 比如分类任务可以使用 CrossEntropyLoss，回归任务可以使用 MSELoss 等。

criterion = nn.CrossEntropyLoss()


output = torch.tensor([
    [1.5, 0.2, -0.5], 
    [0.1, 2.0, 0.3]  
])

# 3. 定义真实标签
target = torch.tensor([0, 2])

loss = criterion(output, target)

print(f"cross entropy loss: {loss}")

# mse损失函数

criterion = nn.MSELoss()

output = torch.tensor([[200.0], [400.0]])

target = torch.tensor([[220.0], [380.0]])
loss = criterion(output, target)

print(f"mse loss: {loss}")



cross entropy loss: 1.164473295211792
mse loss: 400.0


优化器

In [ ]:
# 用于更新模型参数，下面代码仅作演示，可不运行。

import torch.optim as optim

model = SimpleNet()
optimizer = optim.Adam(model.parameters(), lr=0.001)
inputs = torch.randn((32, 10))
targets = torch.randint(0, 2, (32,))

criterion = nn.CrossEntropyLoss()

# epoch就是训练的轮数
for epoch in range(10):
    # 1. 清空梯度
    optimizer.zero_grad()

    # 2. 前向传播
    outputs = model(inputs)

    # 3. 计算损失
    loss = criterion(outputs, targets)

    # 4. 反向传播
    loss.backward()

    # 5. 更新参数
    optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

训练和评估模式

In [ ]:
# 训练，会启用 dropout, batch norm等
model.train()

# 评估，会禁用 dropout, batch norm等
model.eval()

梯度相关的操作

In [ ]:
# backward() 反向传播，计算梯度，其实就是计算 dy/dx，这里y = x^2，所以 dy/dx = 2x
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
y.backward()
print(x.grad)  # tensor([4.])  # 对y = x^2 求导，在x=2时，梯度为4，你也可以修改x的值，验证一下
# detach 分离梯度, 会返回一个新的tensor，这个tensor与原tensor共享数据，但不参与梯度计算，比如在GAN中，判别器和生成器的参数是分开训练的，判别器的梯度只用于更新判别器的参数，而不用于更新生成器的参数。
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
z = y.detach()
print(f"z的具体值: {z}")  # tensor([4.])
print(f"z是否参与梯度计算: {z.requires_grad}")  # False

# detach确实可以禁用某些层的梯度计算，但是如果你是想复用模型参数，做迁移学习最好使用 requires_grad_() 方法
# 比如复用resnet做图像分类任务，你可以将resnet的参数冻结，只训练分类层的参数
import torch
import torchvision.models as models

# 加载一个预训练的 ResNet
resnet = models.resnet18(pretrained=True)
for param in resnet.parameters():
    param.requires_grad_(False)

model.fc = torch.nn.Linear(512, 10)

# 下面两种写法等价，都是只优化分类层的参数
optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 禁用梯度计算, 不希望计算梯度时，可以这样写, with语句的意思是仅在with 语句包裹的块中，梯度计算是禁用的。
# 场景包括模型验证和模型推理，因为在这两种场景下，我们只需要模型的前向传播，而不需要计算梯度。

# 下面的代码仅作演示，需要有真实的 val_loader/test_loader 才能运行

# model.train()
#
# # 模型验证，验证期间不需要更新参数，减少计算量。
# model.eval()
# with torch.no_grad():
#     for data, target in val_loader:
#         output = model(data)
#         val_loss += criterion(output, target).item()
#         # 计算准确率等等
#
# # 模型推理，只要结果，为了极致的速度，最小的内存占用
# model.eval()
# with torch.no_grad():
#     for data, target in test_loader:
#         output = model(data)
#         # 计算准确率等等

容易混淆的点 moddel.eval() 与 torch.no_grad() 的区别
model.eval() 是禁用 dropout, batch norm等层的行为，而 torch.no_grad() 会禁用梯度计算。
| 特性 | model.eval() | torch.no_grad() |
| --- | --- | --- |
| 作用对象 | 层的功能行为 | 梯度计算引擎 |
| 具体影响 | 通知 Dropout 层不要丢弃神经元；通知 BatchNorm 层使用统计均值而非当前 Batch 均值。 | 禁止构建计算图，禁止计算梯度。 |
| 是否省显存 | 否（它依然会构建计算图） | 是（大幅节省显存） |
| 通常用法 | 必须配合使用：在验证/推理时，这两行代码通常是成对出现的。 | 必须配合使用 |


在新版本的pytorch中，会使用 torch.inference_mode() 来替代 torch.no_grad()，会有更极致的性能，在生产环境中使用，具体可参考官方文档。


# 矩阵相关

In [57]:
# 矩阵乘法
a = torch.tensor([[1, 2, 3], [4, 5, 6]])
b = torch.tensor([[7, 8], [9, 10], [11, 12]])

# 下面两种写法等价
print(f"矩阵 @ 结果 = {a @ b}")
print(f"矩阵torch.matmul结果 = {torch.matmul(a, b)}")

# 批量矩阵乘法

A = torch.randn(10, 2, 3)
B = torch.randn(10, 3, 4)

C = torch.matmul(A, B)
print(f"批量矩阵乘法结果: {C.shape}")  # (10, 2, 4)


矩阵 @ 结果 = tensor([[ 58,  64],
        [139, 154]])
矩阵torch.matmul结果 = tensor([[ 58,  64],
        [139, 154]])
批量矩阵乘法结果: torch.Size([10, 2, 4])


In [60]:
# 掩码操作，使用掩码填充至指定值, 通常用于 Attention 机制中，将 padding 部分的注意力分数设为一个非常小的值，从而在计算 softmax 时忽略这些位置。

x = torch.randn(2, 3)
print(f"原始矩阵: {x}")
mask = torch.tensor([[True, False, True], [False, True, False]])

# 将mask为True的位置都填上 -1e9
result = x.masked_fill(mask, -1e9)
print(f"掩码操作结果: {result}")


原始矩阵: tensor([[ 0.2710,  0.4424,  0.0677],
        [-1.5950, -0.2639,  0.4397]])
掩码操作结果: tensor([[-1.0000e+09,  4.4241e-01, -1.0000e+09],
        [-1.5950e+00, -1.0000e+09,  4.3969e-01]])


In [63]:
# 上面是指定位置的掩码，还有上下三角掩码， 这里说的上下是左下方和右上方
# Triangle Lower 是下三角掩码，获取的矩阵是下三角部分，其他位置都是 0。
# Triangle Upper 是上三角掩码，获取的矩阵是上三角部分，其他位置都是 0。

x = torch.ones(4, 4)
print(f"原始矩阵: {x}")
mask = torch.triu(x)
print(f"上三角掩码: {mask}")
mask = torch.tril(x)
print(f"下三角掩码: {mask}")

原始矩阵: tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
上三角掩码: tensor([[1., 1., 1., 1.],
        [0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.]])
下三角掩码: tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


不等于0的位置: tensor([ True, False,  True, False,  True])


# 一个完整的神经网络

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x


model = SimpleNet()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

x = torch.randn(32, 10)  # 32个样本，每个样本有10个特征
y = torch.randint(0, 2, (32,))  # 32个样本，每个样本的标签是0或1

model.train()

optimizer.zero_grad()  # 梯度清零
output = model(x)
loss = criterion(output, y)  # 计算损失
loss.backward()  # 计算梯度
optimizer.step()  # 反向传播，更新参数

print(f"损失值: {loss.item()}")

